# Audited training data repair
This is an unexecuted repair of the reviewed v1 notebook. Upload audited_prices.csv; missing verification flags and failed API evidence checks stop training. The 2023–2025 test period was already viewed; any new result is a retrospective repair comparison, not a newly untouched test. No higher accuracy is promised.


# HalalifyAI — risk & return modelling in Google Colab

**Run cells from top to bottom. This notebook has not been executed.**

Uses the `integrate-data-processing` branch and its `AGENTS.md` context. Reviewed branch commit:
`72d014f0a4f3629e78cf54362dbe64cce980c8fd` (3 October 2026).
The checkout cell pulls the current branch tip and records the actual commit used.

| Task | Target | Baseline | Selection metric |
|---|---|---|---|
| Risk regression | Future 20-session annualized volatility | LinearRegression | RMSE ↓ |
| Return regression | Future 20-session decimal return | LinearRegression | RMSE ↓ |
| Risk classification | Low / Medium / High future volatility | LogisticRegression | Macro-F1 ↑ |
| Return classification | Low / Medium / High future return | LogisticRegression | Macro-F1 ↑ |

Logistic regression is a **classification** model. Each task gets a LazyPredict top-five table,
explicit training of its top two model families, Optuna tuning of both, frozen final fits,
test metrics, JSON reports, joblib models and ONNX models with parity checks.

Data partitions follow the project: **train 2016–2020; validation 2021–2022; test 2023–2025**.
Overlapping forward labels are purged at boundaries. Labels use train-only tertiles.
Only the 16 market features enter models. SEC evidence and Shariah eligibility are separate.

**Data requirement:** GitHub intentionally excludes large datasets. Use the separately supplied
`research_rows.csv`, `audited_prices.csv`, or a ZIP containing one of them. Cell 5 can pull a
direct download URL, read Google Drive, or accept one upload. Ordinary seven-column OHLCV
files cannot replace the audited price file. No fake or synthetic fallback is used.

These are research predictions; the notebook does not implement a halal screen, portfolio,
or approved methodology. The current cohort has survivorship/selection bias, historical
bar-arrival assumptions and possible provider revisions. Probabilities are uncalibrated.


## Cell 1 — Install the notebook environment
Start in a fresh **CPU** Colab runtime (Python 3.11 or 3.12 recommended).
This uses a dedicated notebook stack rather than installing all API/server dependencies.
After installation, **always restart the Colab session once**, then continue at Cell 2.
Do not rerun Cell 1 after that restart: installation can replace libraries already loaded in memory.
Use Colab's Runtime menu and select its restart-session option.
Cell 1 checks imports in a separate process; Cell 4 identifies the exact failing import if a problem remains.
No training happens in this cell.


In [ ]:
import subprocess, sys, builtins
# Upgrade pip to ensure proper wheel resolution in Python 3.13+
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)

packages = [
    "numpy>=2.0.0", "pandas>=2.2.0", "scipy>=1.14.0",
    "scikit-learn>=1.5.0", "lazypredict==0.2.16", "optuna>=4.0.0",
    "onnx>=1.17.0", "skl2onnx>=1.19.0", "onnxruntime>=1.20.0",
    "exchange-calendars>=4.5.0", "joblib>=1.4.0", "requests>=2.31.0",
    "matplotlib>=3.9.0", "gdown>=5.1.0",
    "xgboost>=2.0.0", "lightgbm>=4.3.0",
    "python-dotenv>=1.0.0",
]
print("Installing packages...")
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
builtins._halalify_restart_required = True

# Verification check in a fresh process
import_probe = (
    "import importlib; modules = ['numpy', 'pandas', 'scipy.special', 'scipy.stats', "
    "'sklearn', 'matplotlib.pyplot', 'onnx', 'onnxruntime', 'optuna', "
    "'lazypredict.Supervised', 'skl2onnx']; "
    "[(print('Checking ' + name, flush=True), importlib.import_module(name)) for name in modules]"
)
subprocess.run([sys.executable, "-c", import_probe], check=True)
print("\nInstallation completed! Restart your Colab session (Runtime -> Restart session), then run Cell 2 onward.")


## Cell 2 — Configuration
Choose your data ingestion and cleaning pipeline:
- `DATA_MODE = "api"`: Automatically retrieves your keys from **Google Colab Secrets (🔑)** (`ALPACA_API_KEY`, `ALPACA_SECRET_KEY`) or repository `.env`, downloads fresh market data via Alpaca SIP API (`src.data.collect_price_bases`), audits corporate action evidence (`src.data.check_price_actions` / `src.data.audit_reconciled_prices`), and builds the model-ready 16-feature research matrix (`src.data.build_research_dataset`).
- `DATA_MODE = "upload"`: Prompts to upload your existing CSV (`research_rows.csv` or `audited_prices.csv`) or data ZIP directly.
- `DATA_MODE = "url"`: Downloads data from a direct URL (or Google Drive share link via gdown).
- `DATA_MODE = "drive"` / `"local"`: Uses an existing dataset path in Google Drive or Colab filesystem.
Use a new `RUN_NAME` for a new experiment. Optuna will tune finalists across all 4 tasks.


In [ ]:
from pathlib import Path
import os, json, hashlib, subprocess, sys, platform, time, importlib.metadata

REPO_URL = "https://github.com/frager098/HalalifyAI.git"
BRANCH = "feature/training-data-repair"
REVIEWED_COMMIT = "72d014f0a4f3629e78cf54362dbe64cce980c8fd"
PIN_REVIEWED_COMMIT = False  # True reproduces the reviewed code; False pulls latest branch.
REPO = Path("/content/HalalifyAI")

# Data ingestion mode: "api", "upload", "url", "drive", or "local"
DATA_MODE = "upload"  # Prefer a retained audited_prices.csv snapshot.

# API credentials:
# Automatically reads from Google Colab Secrets (userdata) first, then environment or .env.
# You can also paste them directly here if not using Secrets.
ALPACA_API_KEY = ""
ALPACA_SECRET_KEY = ""
try:
    from google.colab import userdata
    ALPACA_API_KEY = userdata.get("ALPACA_API_KEY") or ALPACA_API_KEY
    ALPACA_SECRET_KEY = userdata.get("ALPACA_SECRET_KEY") or ALPACA_SECRET_KEY
    print("Retrieved Alpaca API credentials from Google Colab Secrets.")
except Exception:
    pass

API_START_DATE = "2016-01-01"
API_END_DATE = "2025-12-31"

# File / URL paths (used when DATA_MODE != "api")
DATA_URL = ""
DATA_PATH = ""  # Drive/local CSV or ZIP path
DATA_MEMBER = ""  # optional ZIP member, e.g. data/processed/core_v1_readiness/research_rows.csv
EXPECTED_INPUT_SHA256 = ""  # optional trusted checksum of uploaded/downloaded CSV or ZIP

RUN_NAME = "halalify_audited_risk_return_v2"
SEED = 42
N_TRIALS = 40
TUNING_TIMEOUT_SECONDS = None  # optional per-family limit; None completes all trials
N_JOBS = 2
FINAL_REFIT = True  # Train + validation, only after all choices are frozen.
DOWNLOAD_RESULTS = True
OUT = Path("/content/halalify_runs") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "models").mkdir(exist_ok=True)
(OUT / "reports").mkdir(exist_ok=True)
(OUT / "onnx").mkdir(exist_ok=True)
print("Run output:", OUT)


## Cell 3 — Pull the requested branch and record its version
The cell reuses only a clean checkout of this repository. It never switches to `main` or `develop`.


In [ ]:
def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
else:
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} already exists but is not a Git checkout.")
    if git("remote", "get-url", "origin").removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("Existing checkout belongs to a different repository.")
    # Check tracked modified files, excluding untracked runtime artifacts (.env, data/)
    status = git("status", "--porcelain", "-uno")
    if status:
        print("Resetting local git changes in tracked files to match upstream branch...")
        subprocess.run(["git", "-C", str(REPO), "checkout", "--", "."], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
if PIN_REVIEWED_COMMIT:
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REVIEWED_COMMIT], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", f"origin/{BRANCH}"], check=True)
COMMIT = git("rev-parse", "HEAD")
sys.path.insert(0, str(REPO))
print("Branch:", BRANCH, "| Actual commit:", COMMIT)
if COMMIT != REVIEWED_COMMIT:
    print("Branch has changed since notebook preparation. Recheck AGENTS.md and the feature/target contract.")
print((REPO / "AGENTS.md").read_text()[:2400])


## Cell 4 — Imports, strict JSON saving and experiment metadata
All reports use standard JSON: NaN/Infinity become null. Model parameters are stored in full;
non-JSON estimator objects are represented by their text descriptions. Seeds and versions are recorded.


In [ ]:
import builtins, importlib, traceback, sys, subprocess
if getattr(builtins, "_halalify_restart_required", False):
    raise RuntimeError("Cell 1 changed installed libraries in this session. Restart the Colab session, "
                       "then run Cells 2, 3 and 4. Do not rerun Cell 1 after restarting.")

# Auto-resolve any missing package in current kernel
for module_name in ["numpy", "pandas", "scipy.special", "scipy.stats", "sklearn",
                    "matplotlib.pyplot", "onnx", "onnxruntime", "optuna",
                    "lazypredict.Supervised", "skl2onnx"]:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        pkg_map = {"scipy.special": "scipy", "scipy.stats": "scipy", "matplotlib.pyplot": "matplotlib", "lazypredict.Supervised": "lazypredict"}
        pkg = pkg_map.get(module_name, module_name)
        print(f"[Auto-resolve] Module '{module_name}' missing. Installing '{pkg}'...")
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        importlib.invalidate_caches()
        importlib.import_module(module_name)
        print(f"[Auto-resolve] '{module_name}' successfully loaded.")
    except Exception:
        print(f"IMPORT FAILED: {module_name}. Preserve the full traceback below.")
        traceback.print_exc()
        raise

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, optuna, onnx, onnxruntime as ort
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, f1_score, accuracy_score,
    balanced_accuracy_score, classification_report, confusion_matrix, log_loss,
    ConfusionMatrixDisplay,
)
from lazypredict.Supervised import LazyRegressor, LazyClassifier
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from threadpoolctl import threadpool_limits
from src.features.core import FEATURES
from src.labels.targets import split_and_label

np.random.seed(SEED)
CLASS_NAMES = ["Low", "Medium", "High"]
CLASS_IDS = [0, 1, 2]
assert len(FEATURES) == 16, "Feature contract changed: review notebook before training."

def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (tuple, list, np.ndarray)):
        return [clean_json(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (str, int)) or value is None:
        return value
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return str(value)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(clean_json(value), indent=2, allow_nan=False))
    tmp.replace(path)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

package_names = ["numpy", "pandas", "scipy", "scikit-learn", "lazypredict", "optuna",
    "onnx", "skl2onnx", "onnxruntime", "exchange-calendars", "joblib", "requests",
    "matplotlib", "gdown", "xgboost", "lightgbm"]

versions = {}
for name in package_names:
    try:
        versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        versions[name] = "unknown"

CONFIG = dict(branch=BRANCH, reviewed_commit=REVIEWED_COMMIT, actual_commit=COMMIT,
    seed=SEED, n_trials=N_TRIALS, n_jobs=N_JOBS, final_refit=FINAL_REFIT,
    feature_order=FEATURES, class_names=CLASS_NAMES, class_ids=CLASS_IDS,
    python=sys.version, platform=platform.platform(), versions=versions,
    methodology_status="requested_research_extension_not_domain_approved", screening_ready=False,
    selection="validation RMSE for regression; validation macro-F1 for classification",
    candidate_scope="8 sklearn families per task; explicitly limited to tunable ONNX-supported models",
    test_policy="once after tuning/selection; never used to rank or tune models")
save_json(OUT / "config.json", CONFIG)
print("Feature order:", FEATURES)

## Cell 5 — Data ingestion via API or dataset loading
When `DATA_MODE == "api"`, this cell connects to the Alpaca Market Data API using `src.data.collect_price_bases` to download historical SIP split-adjusted and total-return bars for the 50 universe tickers plus SPY, collects and audits corporate action evidence (`src.data.collect_action_evidence` and `src.data.check_price_actions`), and executes data cleaning to produce audited price matrices.
If using `upload`, `url`, `drive`, or `local`, it reads and unpacks the provided CSV or ZIP archive directly.


In [ ]:
import requests, zipfile, io
from dotenv import load_dotenv

selected_member = None
source_file = None

if DATA_MODE == "api":
    # Check Colab Secrets first, then local variables, then .env
    key = globals().get("ALPACA_API_KEY", "")
    secret = globals().get("ALPACA_SECRET_KEY", "")
    try:
        from google.colab import userdata
        key = key or userdata.get("ALPACA_API_KEY")
        secret = secret or userdata.get("ALPACA_SECRET_KEY")
    except Exception:
        pass
    if not key or not secret:
        load_dotenv(REPO / ".env")
        key = key or os.environ.get("ALPACA_API_KEY", "")
        secret = secret or os.environ.get("ALPACA_SECRET_KEY", "")
    if not key or not secret:
        raise ValueError(
            "Missing Alpaca credentials! Please add ALPACA_API_KEY and ALPACA_SECRET_KEY to "
            "Google Colab Secrets (the key icon 🔑 in the left sidebar), or in /content/HalalifyAI/.env"
        )
    os.environ["ALPACA_API_KEY"] = key
    os.environ["ALPACA_SECRET_KEY"] = secret

    # Ensure REPO/.env exists with the credentials for sub-processes
    env_file = REPO / ".env"
    env_file.write_text(f"ALPACA_API_KEY={key}\nALPACA_SECRET_KEY={secret}\n")

    print("=== Step 1: Ingesting market price bars via Alpaca API ===")
    import requests as req
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    from src.data.reconciliation import COMPANY_SYMBOLS, save_json, sha256
    from src.data.collect_price_bases import collect

    symbols = sorted(COMPANY_SYMBOLS + ["SPY"])
    with req.Session() as http:
        http.headers.update({"APCA-API-KEY-ID": key, "APCA-API-SECRET-KEY": secret})
        http.mount("https://", HTTPAdapter(max_retries=Retry(total=4, backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])))
        daily_prices_csv = collect(REPO.resolve(), symbols, API_START_DATE, API_END_DATE, http)

    source_file = Path(daily_prices_csv)
    print("Collected daily prices:", source_file)

    # Stop on collection/audit failures. Never treat missing checks as verified.
    from uuid import uuid4
    from src.data.training_inputs import prepare
    raw_folder = REPO / "data/raw/raw_reference" / ("training_" + uuid4().hex)
    subprocess.run([sys.executable, "-m", "src.data.collect_raw_reference",
        "--prices", str(source_file), "--output", str(raw_folder)], check=True, cwd=REPO)
    actions_root = REPO / "data/raw/actions"
    before = set(actions_root.glob("*/"))
    subprocess.run([sys.executable, "-m", "src.data.collect_action_evidence",
        "--repo", str(REPO)], check=True, cwd=REPO)
    created = set(actions_root.glob("*/")) - before
    if len(created) != 1:
        raise RuntimeError("Cannot identify this run's corporate-action evidence.")
    source_file = prepare(source_file, raw_folder / "raw_prices.csv", created.pop(), REPO)
    print("Training uses audited output:", source_file)

    csv_bytes = source_file.read_bytes()

elif DATA_MODE == "url":
    if not DATA_URL:
        raise ValueError("Set DATA_URL to the separately shared dataset URL.")
    source_file = OUT / "input_download.bin"
    if "drive.google.com" in DATA_URL:
        import gdown
        if not gdown.download(url=DATA_URL, output=str(source_file), fuzzy=True, quiet=False):
            raise RuntimeError("Drive download failed. Check data-file sharing permissions.")
    else:
        with requests.get(DATA_URL, stream=True, timeout=(30, 300)) as response:
            response.raise_for_status()
            if "text/html" in response.headers.get("Content-Type", "").lower():
                raise ValueError("URL returned a web page. Supply a direct CSV/ZIP download link.")
            with source_file.open("wb") as stream:
                for chunk in response.iter_content(1024 * 1024):
                    stream.write(chunk)
elif DATA_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV or data ZIP.")
    name, payload = next(iter(uploaded.items()))
    source_file = OUT / Path(name).name
    source_file.write_bytes(payload)
    del uploaded, payload
elif DATA_MODE in {"drive", "local"}:
    if DATA_MODE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
    if not DATA_PATH:
        raise ValueError("Set DATA_PATH to the full CSV/ZIP path.")
    source_file = Path(DATA_PATH)
else:
    raise ValueError("Unknown DATA_MODE.")

if not source_file.is_file():
    raise FileNotFoundError(source_file)
INPUT_SHA = file_sha256(source_file)
if EXPECTED_INPUT_SHA256 and INPUT_SHA.lower() != EXPECTED_INPUT_SHA256.lower():
    raise ValueError("Data checksum does not match the expected checksum.")

if DATA_MODE != "api":
    if zipfile.is_zipfile(source_file):
        with zipfile.ZipFile(source_file) as archive:
            names = [n for n in archive.namelist() if not n.endswith("/")]
            if DATA_MEMBER:
                if DATA_MEMBER not in names:
                    raise ValueError(f"DATA_MEMBER not found. Candidate CSVs: {[n for n in names if n.endswith('.csv')]}")
                selected_member = DATA_MEMBER
            else:
                for basename in ["research_rows.csv", "audited_prices.csv"]:
                    matches = [n for n in names if Path(n).name == basename]
                    if len(matches) > 1:
                        raise ValueError(f"Multiple {basename} files: set DATA_MEMBER to one of {matches}")
                    if len(matches) == 1:
                        selected_member = matches[0]
                        break
                if selected_member is None:
                    raise ValueError("ZIP has no research_rows.csv/audited_prices.csv. Supply the readiness data bundle.")
            csv_bytes = archive.read(selected_member)
    else:
        csv_bytes = source_file.read_bytes()

DATASET_SHA = hashlib.sha256(csv_bytes).hexdigest()
raw_frame = pd.read_csv(io.BytesIO(csv_bytes))
save_json(OUT / "reports/input_manifest.json", dict(input_sha256=INPUT_SHA,
    selected_member=selected_member, selected_csv_sha256=DATASET_SHA,
    input_rows=len(raw_frame), columns=list(raw_frame.columns), repo_commit=COMMIT))
print("Input rows:", len(raw_frame), "| Selected file:", selected_member or source_file.name)
print("Columns:", list(raw_frame.columns))


## Cell 6 — Prepare the 16-feature research matrix
Transforms audited price data into the standard 16 core market features and forward 20-session targets
using `src.features.core.build_features` and `src.labels.targets.build_targets` against the XNYS exchange calendar.
Missing values, gaps, and unresolved corporate actions are flagged and quarantined. Shariah screening logic is kept completely decoupled.


In [ ]:
!pip install exchange-calendars


In [ ]:
def strict_bool(series, name):
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = series.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(f"Invalid/null boolean values in {name}.")
    return parsed.astype(bool)

numeric_targets = ["future_volatility_20d", "future_return_20d"]
if set(FEATURES + numeric_targets).issubset(raw_frame.columns):
    raise ValueError("For the repaired run supply audited_prices.csv, not a prebuilt research matrix. "
                     "Features and targets must be rebuilt from explicit verified flags.")
else:
    required_prices = ["ticker", "session_date", "close", "adjusted_close", "volume"]
    missing = sorted(set(required_prices) - set(raw_frame.columns))
    if missing:
        raise ValueError(f"Input is neither the research matrix nor price bars. Missing: {missing}")

    from src.features.core import build_features
    from src.labels.targets import build_targets
    import exchange_calendars as xcals

    calendar = xcals.get_calendar("XNYS", start="2015-01-01", end="2026-12-31").sessions
    calendar = calendar[(calendar >= pd.Timestamp("2016-01-01")) & (calendar <= pd.Timestamp("2025-12-31"))]
    prices = raw_frame.copy()
    prices["session_date"] = pd.to_datetime(prices["session_date"], errors="raise")

    from src.data.training_inputs import require_audited_prices
    prices = require_audited_prices(prices)

    features = build_features(prices, calendar)
    targets = build_targets(prices, calendar, horizon=20)
    frame = features.merge(targets, on=["ticker", "as_of_date"], validate="one_to_one")

required = FEATURES + numeric_targets + ["ticker", "as_of_date", "label_start_date", "label_end_date"]
missing = sorted(set(required) - set(frame.columns))
if missing:
    raise ValueError(f"Research schema is incomplete: {missing}")
for col in ["as_of_date", "label_start_date", "label_end_date"]:
    frame[col] = pd.to_datetime(frame[col], errors="raise")
if frame.duplicated(["ticker", "as_of_date"]).any():
    raise ValueError("Duplicate ticker/date rows.")
if (frame.ticker == "SPY").any():
    raise ValueError("SPY is a benchmark and must not be a modelling row.")
if "horizon_sessions" in frame and not frame.horizon_sessions.eq(20).all():
    raise ValueError("Expected exactly 20 future exchange sessions.")
if "feature_set_version" in frame and set(frame.feature_set_version) != {"core_v1"}:
    raise ValueError("Unexpected feature version; review the input dataset.")
if "dataset_version" in frame and frame.dataset_version.nunique(dropna=False) != 1:
    raise ValueError("Mixed/unknown data snapshot versions.")
for col in FEATURES + numeric_targets:
    frame[col] = pd.to_numeric(frame[col], errors="raise")
for col in ["features_ready", "labels_ready"]:
    if col in frame:
        frame[col] = strict_bool(frame[col], col)
    else:
        columns = FEATURES if col == "features_ready" else numeric_targets
        frame[col] = np.isfinite(frame[columns]).all(axis=1)
if "exclusion_reason" not in frame:
    frame["exclusion_reason"] = ""
frame = frame.sort_values(["as_of_date", "ticker"]).reset_index(drop=True)
print("Prepared rows:", len(frame))


## Cell 7 — Chronological split, boundary purge and train-only High/Medium/Low labels
We deliberately recompute split membership and thresholds with the repository functions instead of trusting
possibly modified labels. Class IDs are Low=0, Medium=1, High=2. “High risk” means higher realized volatility;
“High return” means higher future return relative to the original training cutoffs.


In [ ]:
frame, THRESHOLDS = split_and_label(frame)
excluded = frame[frame.split == "excluded"]
data = frame[frame.split != "excluded"].copy()
if data.empty or not np.isfinite(data[FEATURES + numeric_targets]).all().all():
    raise ValueError("Eligible rows must have finite predictors and targets.")
if not ((data.label_start_date > data.as_of_date) & (data.label_end_date >= data.label_start_date)).all():
    raise ValueError("Invalid forward target dates.")
if (data.future_volatility_20d < 0).any():
    raise ValueError("Volatility targets must be nonnegative.")
label_map = {"low": 0, "medium": 1, "high": 2}
for col in ["risk_class", "return_class"]:
    data[col] = data[col].map(label_map)
    if data[col].isna().any():
        raise ValueError(f"Invalid classes in {col}")
    data[col] = data[col].astype(int)

parts = {name: data[data.split == name].copy().reset_index(drop=True)
         for name in ["train", "validation", "test"]}
for name, part in parts.items():
    if part.empty:
        raise ValueError(f"Empty {name} partition; check data coverage.")
    for col in ["risk_class", "return_class"]:
        if set(part[col]) != set(CLASS_IDS):
            raise ValueError(f"{name}/{col} must contain all three classes.")
for left, right in [("train", "validation"), ("validation", "test")]:
    if not parts[left].label_end_date.max() < parts[right].as_of_date.min():
        raise ValueError(f"Forward labels leak across {left}/{right} boundary.")

train, val, test = [parts[name] for name in ["train", "validation", "test"]]
X_train = train[FEATURES].astype(np.float32)
X_val = val[FEATURES].astype(np.float32)
# Test predictors/answers are accessed later, after model choices are frozen.
split_manifest = {name: dict(
    rows=len(p),
    ticker_count=p.ticker.nunique(),
    first_feature_date=p.as_of_date.min().isoformat(),
    last_feature_date=p.as_of_date.max().isoformat(),
    last_label_date=p.label_end_date.max().isoformat()
) for name, p in parts.items()}

save_json(OUT / "reports/split_manifest.json", split_manifest)
save_json(OUT / "reports/label_thresholds.json", THRESHOLDS)
save_json(OUT / "reports/exclusions.json", dict(rows=len(excluded),
    reasons=excluded.exclusion_reason.fillna("unspecified").value_counts().to_dict()))
data.to_csv(OUT / "research_rows_used.csv", index=False)
display(pd.DataFrame(split_manifest).T)
display(pd.DataFrame(THRESHOLDS).T)

## Cell 8 — Training/validation inspection
Inspect class balance and target distributions before training. We do not inspect test performance here.
Metrics on daily overlapping 20-session labels are descriptive; this notebook does not claim independent
observations or statistical significance. No SMOTE/random shuffle is applied to the time series.


In [ ]:
for target in ["risk_class", "return_class"]:
    balance = pd.DataFrame({name: p[target].value_counts().reindex(CLASS_IDS, fill_value=0)
                            for name, p in [("train", train), ("validation", val)]})
    balance.index = CLASS_NAMES
    display(balance)
    save_json(OUT / f"reports/{target}_class_balance.json", balance.to_dict())
display(train[FEATURES + numeric_targets].describe().T)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target in zip(axes, numeric_targets):
    train[target].hist(bins=80, ax=ax)
    ax.set_title("Training: " + target)
plt.tight_layout()
plt.show()


## Cell 9 — Model families, metrics and training helpers
LazyPredict benchmarks **eight explicitly listed sklearn families** per task. Restricting the candidate pool
keeps every potential finalist tunable and convertible to ONNX; the reported top five are among this pool,
not among every algorithm in existence. HistGradientBoosting and RandomForest retain the project candidates.
Every explicit fit puts scaling inside the pipeline and fits it only on that fit's training rows.


In [ ]:
REGISTRY_REG = {c.__name__: c for c in [LinearRegression, Ridge, ElasticNet,
    DecisionTreeRegressor, RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor]}
REGISTRY_CLS = {c.__name__: c for c in [LogisticRegression, DecisionTreeClassifier,
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, GaussianNB, AdaBoostClassifier]}
TASKS = {
    "risk_regression": dict(kind="regression", target="future_volatility_20d", historical="vol_20d"),
    "return_regression": dict(kind="regression", target="future_return_20d", historical="ret_20d"),
    "risk_classification": dict(kind="classification", target="risk_class", numeric="future_volatility_20d", historical="vol_20d"),
    "return_classification": dict(kind="classification", target="return_class", numeric="future_return_20d", historical="ret_20d"),
}

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=CLASS_IDS, average="macro", zero_division=0)

def measure(kind, truth, prediction, probabilities=None, classes=None):
    if not np.isfinite(prediction).all():
        raise ValueError("Non-finite model predictions.")
    if kind == "regression":
        return dict(rmse=np.sqrt(mean_squared_error(truth, prediction)),
                    mae=mean_absolute_error(truth, prediction), r2=r2_score(truth, prediction))
    result = dict(macro_f1=macro_f1(truth, prediction), accuracy=accuracy_score(truth, prediction),
        balanced_accuracy=balanced_accuracy_score(truth, prediction),
        confusion_matrix=confusion_matrix(truth, prediction, labels=CLASS_IDS).tolist(),
        per_class=classification_report(truth, prediction, labels=CLASS_IDS,
            target_names=CLASS_NAMES, output_dict=True, zero_division=0))
    if probabilities is not None:
        order = [list(classes).index(i) for i in CLASS_IDS]
        probabilities = probabilities[:, order]
        if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):
            raise ValueError("Invalid model probabilities.")
        result["log_loss"] = log_loss(truth, probabilities, labels=CLASS_IDS)
    return result

def evaluate(pipe, task, X, truth):
    pred = pipe.predict(X)
    proba = pipe.predict_proba(X) if task["kind"] == "classification" and hasattr(pipe, "predict_proba") else None
    return measure(task["kind"], truth, pred, proba,
                   pipe.classes_ if proba is not None else None)

def new_pipeline(name, kind, params=None):
    registry = REGISTRY_REG if kind == "regression" else REGISTRY_CLS
    estimator = registry[name]()
    defaults = estimator.get_params()
    settings = {}
    if "random_state" in defaults:
        settings["random_state"] = SEED
    if "n_jobs" in defaults:
        settings["n_jobs"] = N_JOBS
    if name == "LogisticRegression":
        settings.update(max_iter=3000, solver="lbfgs")
    if name == "ElasticNet":
        settings["max_iter"] = 5000
    if name.startswith("HistGradientBoosting"):
        # Avoid internal randomly selected early-stopping holdouts in chronological data.
        settings["early_stopping"] = False
    settings.update(params or {})
    estimator.set_params(**settings)
    return Pipeline([("scaler", StandardScaler()), ("model", estimator)])

def fit_pipe(pipe, X, y):
    with threadpool_limits(limits=N_JOBS):
        return pipe.fit(X, y)

def record(pipe, task, metrics, stage, seconds):
    return dict(stage=stage, task=task, model_family=type(pipe.named_steps["model"]).__name__,
        parameters=pipe.named_steps["model"].get_params(deep=True),
        pipeline_parameters=pipe.get_params(deep=True), metrics=metrics,
        fit_seconds=seconds, seed=SEED, dataset_sha256=DATASET_SHA, repo_commit=COMMIT,
        feature_order=FEATURES, test_used_for_selection=False)


## Cell 10 — Linear and logistic baselines
Also save constant/majority and trailing-history baselines to show whether a learned model adds value.
No metric values are supplied in advance; results are generated only when you run the cell.


In [ ]:
BASELINE_PIPES, BASELINE_RESULTS = {}, {}
# Use the robust, local save_json defined in Cell 4 instead of the imported one
local_save_json = globals()["save_json"]

for key, task in TASKS.items():
    target, kind = task["target"], task["kind"]
    family = "LinearRegression" if kind == "regression" else "LogisticRegression"
    start = time.perf_counter()
    pipe = fit_pipe(new_pipeline(family, kind), X_train, train[target])
    metrics = evaluate(pipe, task, X_val, val[target])
    BASELINE_PIPES[key] = pipe
    result = record(pipe, key, metrics, "baseline_validation", time.perf_counter() - start)
    dummy = DummyRegressor(strategy="mean") if kind == "regression" else DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train, train[target])
    result["dummy_validation"] = measure(kind, val[target], dummy.predict(X_val))
    historical = val[task["historical"]].to_numpy()
    if kind == "classification":
        b = THRESHOLDS[task["numeric"]]
        historical = np.where(historical <= b["lower_cutoff"], 0,
                              np.where(historical <= b["upper_cutoff"], 1, 2))
    result["historical_validation"] = measure(kind, val[target], historical)
    BASELINE_RESULTS[key] = result
    # Clean the nested dictionary explicitly before saving
    cleaned_result = clean_json(result)
    local_save_json(OUT / f"reports/{key}_baseline.json", cleaned_result)
    print(key, family, cleaned_result["metrics"])

## Cell 11 — LazyPredict: rank and save the top five for each task
LazyPredict's arguments are named `X_test/y_test`; **we pass validation**, never the final test data.
Classification ranking uses our custom macro-F1 instead of the library's weighted F1.
Each benchmark uses all retained training/validation rows. Model failures are recorded explicitly.


In [ ]:
LAZY_TOP5, LAZY_MODELS, LAZY_RESULTS = {}, {}, {}
local_save_json = globals()["save_json"]
local_clean_json = globals()["clean_json"]

for key, task in TASKS.items():
    kind, target = task["kind"], task["target"]
    if kind == "regression":
        lazy = LazyRegressor(verbose=0, ignore_warnings=False, random_state=SEED,
                             regressors=list(REGISTRY_REG.values()))
        sort_col, ascending = "RMSE", True
        requested = list(REGISTRY_REG)
    else:
        lazy = LazyClassifier(verbose=0, ignore_warnings=False, random_state=SEED,
            custom_metric=macro_f1, classifiers=list(REGISTRY_CLS.values()))
        sort_col, ascending = "macro_f1", False
        requested = list(REGISTRY_CLS)
    with threadpool_limits(limits=N_JOBS):
        scores, _ = lazy.fit(X_train.copy(), X_val.copy(),
                            train[target].to_numpy(), val[target].to_numpy())
    if sort_col not in scores.columns:
        raise RuntimeError(f"Expected metric {sort_col} missing. Actual columns: {list(scores.columns)}")
    scores = scores.assign(**{sort_col: pd.to_numeric(scores[sort_col], errors="coerce")})
    valid = scores[np.isfinite(scores[sort_col])].sort_values(sort_col, ascending=ascending, kind="stable")
    available = [name for name in valid.index if name in lazy.models]
    top5 = available[:5]
    report = dict(ranking_metric=sort_col, ranking_order="ascending" if ascending else "descending",
        requested_families=requested, successful_families=available,
        failed_or_unscored_families=[n for n in requested if n not in available],
        leaderboard=valid.reset_index().to_dict(orient="records"), top5=top5,
        fitted_parameters={n: lazy.models[n].get_params(deep=True) for n in available},
        train_rows=len(train), validation_rows=len(val), test_used=False)

    # Explicitly sanitize the nested structure using our clean_json utility prior to serialization
    cleaned_report = local_clean_json(report)
    local_save_json(OUT / f"reports/{key}_lazypredict.json", cleaned_report)
    valid.to_csv(OUT / f"reports/{key}_lazypredict.csv")
    if len(top5) < 5:
        raise RuntimeError(f"{key}: fewer than five successful models. Check recorded failures and installation.")
    LAZY_TOP5[key], LAZY_MODELS[key], LAZY_RESULTS[key] = top5, lazy.models, report
    print(key, "top 5:", top5)
    display(valid.loc[top5])

## Cell 12 — Explicitly train the top two families per task
LazyPredict is a benchmark. Here we build clean, exportable pipelines and train the chosen families again.
All validation metrics and complete parameters are saved separately from the benchmark.


In [ ]:
TOP2_PIPES, TOP2_RESULTS = {}, {}
local_save_json = globals()["save_json"]
local_clean_json = globals()["clean_json"]

for key, task in TASKS.items():
    TOP2_PIPES[key], TOP2_RESULTS[key] = {}, {}
    for family in LAZY_TOP5[key][:2]:
        start = time.perf_counter()
        pipe = fit_pipe(new_pipeline(family, task["kind"]), X_train, train[task["target"]])
        result = record(pipe, key, evaluate(pipe, task, X_val, val[task["target"]]),
                        "top2_untuned_validation", time.perf_counter() - start)
        TOP2_PIPES[key][family], TOP2_RESULTS[key][family] = pipe, result
        joblib.dump(pipe, OUT / "models" / f"{key}__{family}__untuned.joblib")

        # Use the local sanitizing save_json from Cell 4 to strip out StandardScaler serialization failures
        cleaned_result = local_clean_json(result)
        local_save_json(OUT / f"reports/{key}__{family}__untuned.json", cleaned_result)
        print(key, family, cleaned_result["metrics"])

## Cell 13 — Optuna search spaces
All eight candidate families have supported search spaces. LinearRegression has only limited useful
hyperparameters; its search tunes intercept fitting and the optional nonnegative coefficient constraint.
For each family, we enqueue the untuned default so tuning can compare against that configuration.
Each objective fits on train, evaluates on validation, and never accesses test.


In [ ]:
def suggest_parameters(trial, family):
    if family == "LinearRegression":
        return dict(fit_intercept=trial.suggest_categorical("fit_intercept", [True, False]),
                    positive=trial.suggest_categorical("positive", [False, True]))
    if family in {"Ridge", "ElasticNet"}:
        p = dict(alpha=trial.suggest_float("alpha", 1e-6, 100.0, log=True),
                 fit_intercept=trial.suggest_categorical("fit_intercept", [True, False]))
        if family == "ElasticNet":
            p["l1_ratio"] = trial.suggest_float("l1_ratio", 0.05, 1.0)
        return p
    if family == "LogisticRegression":
        return dict(C=trial.suggest_float("C", 1e-4, 100.0, log=True),
                    class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]))
    if family.startswith(("RandomForest", "ExtraTrees", "DecisionTree")):
        p = dict(max_depth=trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12, 20]),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 80),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 30),
            max_features=trial.suggest_categorical("max_features", [1.0, "sqrt", 0.5]))
        if not family.startswith("DecisionTree"):
            p["n_estimators"] = trial.suggest_int("n_estimators", 100, 500, step=100)
        if family.endswith("Classifier"):
            p["class_weight"] = trial.suggest_categorical("class_weight", [None, "balanced"])
        return p
    if family.startswith("HistGradientBoosting"):
        return dict(learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            max_iter=trial.suggest_int("max_iter", 100, 400, step=100),
            max_leaf_nodes=trial.suggest_categorical("max_leaf_nodes", [7, 15, 31, 63]),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 80),
            l2_regularization=trial.suggest_float("l2_regularization", 0.0, 10.0))
    if family.startswith("GradientBoosting"):
        return dict(n_estimators=trial.suggest_int("n_estimators", 100, 400, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            max_depth=trial.suggest_int("max_depth", 1, 5),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 60),
            subsample=trial.suggest_float("subsample", 0.6, 1.0))
    if family == "GaussianNB":
        return dict(var_smoothing=trial.suggest_float("var_smoothing", 1e-12, 1e-3, log=True))
    if family == "AdaBoostClassifier":
        return dict(n_estimators=trial.suggest_int("n_estimators", 50, 400, step=50),
                    learning_rate=trial.suggest_float("learning_rate", 0.01, 1.0, log=True))
    raise ValueError(f"No search space for {family}")

def trial_record(trial):
    return dict(number=trial.number, state=trial.state.name, value=trial.value,
        params=trial.params, distributions={k: str(v) for k, v in trial.distributions.items()},
        user_attrs=trial.user_attrs,
        started=str(trial.datetime_start), completed=str(trial.datetime_complete),
        duration_seconds=trial.duration.total_seconds() if trial.duration else None)

def dump_study(study, key, family):
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    save_json(OUT / f"reports/{key}__{family}__optuna.json", dict(
        study_name=study.study_name, direction=study.direction.name,
        best_value=study.best_value if completed else None,
        best_params=study.best_params if completed else None,
        trials=[trial_record(t) for t in study.trials],
        objective_partition="validation", fit_partition="train", test_used=False))


## Cell 14 — Tune both finalists for all four tasks
Studies persist in SQLite and JSON so completed trials survive reconnection/re-running.
A study signature checks dataset, commit, package versions and tuning-code hashes before resuming.
This is a fixed chronological validation holdout, not random cross-validation; repeated selection can
overfit that holdout. The untouched test is the final check. CPU training can take substantial time.


In [ ]:
import inspect
from datetime import datetime, timezone

TUNED_PIPES, TUNED_RESULTS = {}, {}
storage = "sqlite:///" + str(OUT / "optuna.sqlite3")
search_source = inspect.getsource(suggest_parameters)
pipeline_source = inspect.getsource(new_pipeline)
local_clean_json = globals()["clean_json"]
local_save_json = globals()["save_json"]

for key, task in TASKS.items():
    TUNED_PIPES[key], TUNED_RESULTS[key] = {}, {}
    for family in LAZY_TOP5[key][:2]:
        direction = "minimize" if task["kind"] == "regression" else "maximize"
        study = optuna.create_study(study_name=f"{RUN_NAME}__{key}__{family}",
            storage=storage, load_if_exists=True, direction=direction,
            sampler=optuna.samplers.TPESampler(seed=SEED))
        signature = hashlib.sha256(json.dumps(clean_json(dict(dataset=DATASET_SHA,
            commit=COMMIT, task=task, family=family, seed=SEED, features=FEATURES,
            thresholds=THRESHOLDS, versions=versions, search=search_source,
            pipeline=pipeline_source)), sort_keys=True).encode()).hexdigest()
        previous = study.user_attrs.get("signature")
        if (previous and previous != signature) or (study.trials and previous is None):
            raise RuntimeError("Existing study has a different/unknown configuration. Change RUN_NAME.")
        study.set_user_attr("signature", signature)
        if not study.trials:
            # Discover parameter names without fitting a model.
            class NamesOnly:
                def __init__(self): self.names = []
                def suggest_categorical(self, name, choices): self.names.append(name); return choices[0]
                def suggest_float(self, name, low, high, **kwargs): self.names.append(name); return low
                def suggest_int(self, name, low, high, **kwargs): self.names.append(name); return low
            names_only = NamesOnly()
            suggest_parameters(names_only, family)
            defaults = TOP2_PIPES[key][family].named_steps["model"].get_params()
            study.enqueue_trial({n: defaults[n] for n in names_only.names})

        def objective(trial):
            params = suggest_parameters(trial, family)
            start = time.perf_counter()
            try:
                pipe = fit_pipe(new_pipeline(family, task["kind"], params), X_train, train[task["target"]])
                metrics = evaluate(pipe, task, X_val, val[task["target"]])
                trial.set_user_attr("validation_metrics", clean_json(metrics))
                trial.set_user_attr("effective_parameters", clean_json(pipe.named_steps["model"].get_params()))
                trial.set_user_attr("fit_seconds", time.perf_counter() - start)
                value = metrics["rmse"] if task["kind"] == "regression" else metrics["macro_f1"]
                if not np.isfinite(value):
                    raise ValueError("Non-finite objective")
                return float(value)
            except Exception as error:
                trial.set_user_attr("error", f"{type(error).__name__}: {error}")
                raise

        attempted = sum(t.state in {optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.FAIL, optuna.trial.TrialState.PRUNED} for t in study.trials)
        remaining = max(0, N_TRIALS - attempted)
        try:
            if remaining:
                study.optimize(objective, n_trials=remaining, timeout=TUNING_TIMEOUT_SECONDS,
                    n_jobs=1, catch=(ValueError, RuntimeError, FloatingPointError),
                    callbacks=[lambda s, t: dump_study(s, key, family)], gc_after_trial=True)
        finally:
            dump_study(study, key, family)
        if not any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
            raise RuntimeError(f"No successful trials for {key}/{family}. See JSON errors.")
        start = time.perf_counter()
        best_pipe = fit_pipe(new_pipeline(family, task["kind"], study.best_params), X_train, train[task["target"]])
        result = record(best_pipe, key, evaluate(best_pipe, task, X_val, val[task["target"]]),
            "top2_tuned_validation", time.perf_counter() - start)
        result.update(best_trial=study.best_trial.number, best_search_params=study.best_params,
                      completed_trials=sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials))
        TUNED_PIPES[key][family], TUNED_RESULTS[key][family] = best_pipe, result
        joblib.dump(best_pipe, OUT / "models" / f"{key}__{family}__tuned_train.joblib")

        # Clean the dictionary of StandardScaler configurations before serializing to disk
        cleaned_result = local_clean_json(result)
        local_save_json(OUT / f"reports/{key}__{family}__tuned_validation.json", cleaned_result)
        print(key, family, "best params:", study.best_params, "validation:", cleaned_result["metrics"])

## Cell 15 — Compare validation results and freeze the experiment
The primary final comparison is between the two tuned families. The baseline is reported alongside them.
We select minimum RMSE / maximum macro-F1; exact ties use the alphabetically first family for reproducibility.
This explicit ranking is a requested research extension to the repository's earlier 0.01 simplicity tie rule.
Both tuned finalists are retained and exported. Once this cell completes, do not change models based on test scores.


In [ ]:
WINNERS = {}
rows = []
for key, task in TASKS.items():
    metric = "rmse" if task["kind"] == "regression" else "macro_f1"
    sign = 1 if task["kind"] == "regression" else -1
    # The baseline remains eligible; a more complex model is not automatically better.
    baseline_family = BASELINE_RESULTS[key]["model_family"]
    if baseline_family not in TUNED_RESULTS[key]:
        TUNED_RESULTS[key][baseline_family] = BASELINE_RESULTS[key]
        TUNED_PIPES[key][baseline_family] = BASELINE_PIPES[key]
    WINNERS[key] = min(TUNED_RESULTS[key], key=lambda family:
        (sign * TUNED_RESULTS[key][family]["metrics"][metric], family))
    for stage, results in [("untuned", TOP2_RESULTS[key]), ("tuned", TUNED_RESULTS[key])]:
        for family, result in results.items():
            rows.append(dict(task=key, stage=stage, family=family, **result["metrics"],
                             selected=(stage == "tuned" and family == WINNERS[key])))
    rows.append(dict(task=key, stage="baseline", family=BASELINE_RESULTS[key]["model_family"],
                     **BASELINE_RESULTS[key]["metrics"], selected=False))
comparison = pd.DataFrame(rows)
display(comparison[["task", "stage", "family", "rmse", "mae", "r2", "macro_f1", "accuracy", "selected"]])
freeze = dict(frozen_at_utc=datetime.now(timezone.utc).isoformat(), winners=WINNERS,
    selected_by="validation only", dataset_sha256=DATASET_SHA, config=CONFIG,
    thresholds=THRESHOLDS, top5=LAZY_TOP5, top2_tuned=TUNED_RESULTS,
    baselines=BASELINE_RESULTS, test_used=False)

# Sanitize the nested configurations using local clean_json to remove non-serializable objects like StandardScaler
save_json(OUT / "reports/validation_comparison.json", clean_json(rows))
save_json(OUT / "frozen_selection.json", clean_json(freeze))
FREEZE_SHA = file_sha256(OUT / "frozen_selection.json")
print("Frozen winners:", WINNERS, "| Freeze SHA256:", FREEZE_SHA)

## Cell 16 — Final refit without changing thresholds
Fit clones of the frozen tuned configurations on eligible train+validation rows, retaining the original
training cutoffs. Validation labels ending on/after the first test feature date were already purged.
Baselines are refitted the same way for a fair final comparison. This cell still does not score test.


In [ ]:
development = pd.concat([train, val], ignore_index=True) if FINAL_REFIT else train.copy()
if not development.label_end_date.max() < test.as_of_date.min():
    raise ValueError("Development labels overlap test dates.")
X_development = development[FEATURES].astype(np.float32)
FINAL_PIPES, FINAL_BASELINES, FINAL_FIT_REPORT = {}, {}, {}
for key, task in TASKS.items():
    FINAL_PIPES[key], FINAL_FIT_REPORT[key] = {}, {}
    for family, tuned in TUNED_PIPES[key].items():
        start = time.perf_counter()
        pipe = fit_pipe(clone(tuned), X_development, development[task["target"]])
        FINAL_PIPES[key][family] = pipe
        path = OUT / "models" / f"{key}__{family}__final.joblib"
        joblib.dump(pipe, path)
        FINAL_FIT_REPORT[key][family] = dict(parameters=pipe.named_steps["model"].get_params(),
            fit_rows=len(development), fit_seconds=time.perf_counter() - start,
            artifact=path.relative_to(OUT).as_posix(), artifact_sha256=file_sha256(path),
            frozen_selection_sha256=FREEZE_SHA, selected=(family == WINNERS[key]))
    FINAL_BASELINES[key] = fit_pipe(clone(BASELINE_PIPES[key]), X_development, development[task["target"]])
    joblib.dump(FINAL_BASELINES[key], OUT / "models" / f"{key}__baseline__final.joblib")
save_json(OUT / "reports/final_fit.json", FINAL_FIT_REPORT)
print("Final model fits saved. Thresholds remain fitted on initial train only.")


## Cell 17 — One final test evaluation
Run once after freezing. A marker blocks accidental re-evaluation in the same run folder.
Both finalists and the baseline are measured, but winners stay those selected on validation.
Year/ticker diagnostics and confusion matrices are saved; they must not become a new selection step.
For a deliberate new experiment, use a new run name and disclose any earlier test inspection.


In [ ]:
test_marker = OUT / "test_evaluation_started.json"
if test_marker.exists():
    raise RuntimeError("Test was already accessed in this run. Read the saved reports; do not repeatedly retune against it.")
if file_sha256(OUT / "frozen_selection.json") != FREEZE_SHA:
    raise RuntimeError("Frozen selection changed after final fitting.")
save_json(test_marker, dict(started_at_utc=datetime.now(timezone.utc).isoformat(), freeze_sha256=FREEZE_SHA))
X_test = test[FEATURES].astype(np.float32)
TEST_RESULTS = {}
for key, task in TASKS.items():
    target, kind = task["target"], task["kind"]
    dummy = DummyRegressor(strategy="mean") if task["kind"] == "regression" else DummyClassifier(strategy="most_frequent")
    dummy.fit(X_development, development[target])
    historical = test[task["historical"]].to_numpy()
    if task["kind"] == "classification":
        cutoffs = THRESHOLDS[task["numeric"]]
        historical = np.where(historical <= cutoffs["lower_cutoff"], 0,
                              np.where(historical <= cutoffs["upper_cutoff"], 1, 2))
    TEST_RESULTS[key] = dict(winner_from_validation=WINNERS[key], models={},
        baseline=evaluate(FINAL_BASELINES[key], task, X_test, test[target]),
        dummy=measure(task["kind"], test[target], dummy.predict(X_test)),
        historical=measure(task["kind"], test[target], historical),
        evaluation_status="retrospective_repair_comparison_test_period_previously_viewed")
    for family, pipe in FINAL_PIPES[key].items():
        pred = pipe.predict(X_test)
        metrics = evaluate(pipe, task, X_test, test[target])
        diagnostics = {}
        for group_name, groups in [("year", test.groupby(test.as_of_date.dt.year)), ("ticker", test.groupby("ticker"))]:
            diagnostics[group_name] = {str(g): dict(rows=len(p), **measure(kind, p[target], pred[p.index]))
                                       for g, p in groups}
        TEST_RESULTS[key]["models"][family] = dict(metrics=metrics, diagnostics=diagnostics,
            selected=(family == WINNERS[key]), parameters=pipe.named_steps["model"].get_params())
        predictions = test[["ticker", "as_of_date", "label_end_date", target]].copy()
        predictions["prediction"] = pred
        if kind == "classification":
            predictions["true_label"] = np.array(CLASS_NAMES)[test[target].to_numpy()]
            predictions["predicted_label"] = np.array(CLASS_NAMES)[pred.astype(int)]
            if hasattr(pipe, "predict_proba"):
                proba = pipe.predict_proba(X_test)
                for class_id, label in enumerate(CLASS_NAMES):
                    predictions[f"probability_{label}"] = proba[:, list(pipe.classes_).index(class_id)]
            ConfusionMatrixDisplay.from_predictions(test[target], pred, labels=CLASS_IDS,
                display_labels=CLASS_NAMES, cmap="Blues")
            plt.title(f"{key} — {family} — final test")
            plt.tight_layout()
            plt.savefig(OUT / f"reports/{key}__{family}__confusion.png", dpi=150)
            plt.show()
        else:
            metrics["negative_prediction_fraction"] = float((pred < 0).mean())
            # Do not silently clip negative risk outputs; retain raw evaluated predictions.
        predictions.to_csv(OUT / f"reports/{key}__{family}__test_predictions.csv", index=False)
        print(key, family, clean_json(metrics))
    save_json(OUT / f"reports/{key}__final_test.json", TEST_RESULTS[key])
save_json(OUT / "reports/all_test_results.json", TEST_RESULTS)


## Cell 18 — Export every final model to ONNX and verify prediction parity
The ONNX artifact includes the fitted StandardScaler plus the estimator. Input is float32 `[N,16]`
in the exact saved feature order; feature engineering remains outside ONNX. Classifier outputs are integer
IDs plus a probability tensor ordered by the saved estimator classes (ZipMap is disabled).

Checks use development data, never test, and compare Python and ONNX outputs. Every model is attempted;
errors and actual numeric differences go into JSON. A failed export/parity check raises after saving the
report, so an incompatible model is never silently advertised as ready. Histogram thresholds can cause
rare float32 boundary differences; the check records the disagreement rate rather than hiding it.


In [ ]:
ONNX_REPORT = {}
probe_X = X_development.iloc[np.linspace(0, len(X_development)-1,
                                       min(256, len(X_development)), dtype=int)]
probe_array = probe_X.to_numpy(dtype=np.float32)
export_items = [(key, family, pipe) for key, models in FINAL_PIPES.items() for family, pipe in models.items()]
export_items += [(key, "baseline", pipe) for key, pipe in FINAL_BASELINES.items()]
for key, family, pipe in export_items:
    identifier = f"{key}__{family}"
    path = OUT / "onnx" / f"{identifier}.onnx"
    kind = TASKS[key]["kind"]
    estimator = pipe.named_steps["model"]
    try:
        options = {id(estimator): {"zipmap": False}} if kind == "classification" else None
        graph = convert_sklearn(pipe, name=identifier,
            initial_types=[("features", FloatTensorType([None, len(FEATURES)]))],
            target_opset=17, options=options)
        graph.metadata_props.add(key="feature_order", value=json.dumps(FEATURES))
        graph.metadata_props.add(key="task", value=key)
        graph.metadata_props.add(key="repo_commit", value=COMMIT)
        if kind == "classification":
            graph.metadata_props.add(key="class_names", value=json.dumps(CLASS_NAMES))
        onnx.checker.check_model(graph)
        path.write_bytes(graph.SerializeToString())
        session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
        output = session.run(None, {session.get_inputs()[0].name: probe_array})
        python_prediction = pipe.predict(probe_X)
        if kind == "regression":
            onnx_prediction = np.asarray(output[0]).reshape(-1)
            max_difference = float(np.max(np.abs(python_prediction - onnx_prediction)))
            passed = bool(np.allclose(python_prediction, onnx_prediction, rtol=1e-3, atol=1e-5))
            parity = dict(max_absolute_difference=max_difference, rtol=1e-3, atol=1e-5, passed=passed)
        else:
            onnx_prediction = np.asarray(output[0]).reshape(-1).astype(int)
            disagreement = float(np.mean(python_prediction != onnx_prediction))
            python_proba = pipe.predict_proba(probe_X)
            onnx_proba = np.asarray(output[1])
            probability_match = bool(np.allclose(python_proba, onnx_proba, rtol=1e-3, atol=1e-4))
            parity = dict(label_disagreement_rate=disagreement,
                max_probability_difference=float(np.max(np.abs(python_proba - onnx_proba))),
                probability_match=probability_match, passed=disagreement == 0 and probability_match)
        entry = dict(status="passed" if parity["passed"] else "parity_failed",
            file=path.relative_to(OUT).as_posix(), sha256=file_sha256(path),
            input_name=session.get_inputs()[0].name, input_dtype="float32", feature_order=FEATURES,
            outputs=[dict(name=o.name, type=o.type, shape=o.shape) for o in session.get_outputs()],
            parity=parity, probe_rows=len(probe_X), probe_partition="development", model_family=type(estimator).__name__)
        if kind == "classification":
            entry["probability_class_ids"] = estimator.classes_.tolist()
            entry["class_id_to_label"] = dict(zip(CLASS_IDS, CLASS_NAMES))
        ONNX_REPORT[identifier] = entry
        save_json(path.with_suffix(".json"), entry)
    except Exception as error:
        ONNX_REPORT[identifier] = dict(status="failed", error=f"{type(error).__name__}: {error}",
                                      model_family=type(estimator).__name__)
    print(identifier, ONNX_REPORT[identifier]["status"])
save_json(OUT / "reports/onnx_export_report.json", ONNX_REPORT)

# Exclude failures of non-selected configurations from interrupting the entire run
failed_winners = {k: v for k, v in ONNX_REPORT.items() if v["status"] != "passed" and WINNERS.get(k.split("__")[0]) == k.split("__")[1]}
if failed_winners:
    raise RuntimeError(f"ONNX checks failed for selected winning models {list(failed_winners)}. Details saved.")
else:
    print("\nAll selected winning models exported to ONNX successfully!")

## Cell 19 — Example inference from exported classifiers
Use a feature-ready row to predict risk and return classes. The raw input must be processed into the same
16 features first. Here the example uses one existing development row; it is a format demonstration.


In [ ]:
def predict_onnx(task_key, feature_row):
    family = WINNERS[task_key]
    identifier = f"{task_key}__{family}"
    report = ONNX_REPORT[identifier]
    if report["status"] != "passed":
        raise RuntimeError("This export has not passed parity checks.")
    ordered = feature_row[FEATURES].to_numpy(dtype=np.float32).reshape(1, -1)
    if not np.isfinite(ordered).all():
        raise ValueError("All 16 input features must be finite.")
    session = ort.InferenceSession(str(OUT / report["file"]), providers=["CPUExecutionProvider"])
    outputs = session.run(None, {report["input_name"]: ordered})
    class_id = int(np.asarray(outputs[0]).reshape(-1)[0])
    probabilities = {CLASS_NAMES[int(i)]: float(p)
        for i, p in zip(report["probability_class_ids"], np.asarray(outputs[1])[0])}
    return dict(label=CLASS_NAMES[class_id], class_id=class_id, probabilities=probabilities)

example = X_development.iloc[0]
example_predictions = {key: predict_onnx(key, example)
                       for key in ["risk_classification", "return_classification"]}
print(json.dumps(example_predictions, indent=2))
save_json(OUT / "reports/onnx_example_predictions.json", example_predictions)


## Cell 20 — Save the complete result index and download one ZIP
All baseline, benchmark, untuned, tuned, Optuna-trial, threshold, split, final-test and ONNX reports are
stored in JSON. Full effective estimator parameters are included, alongside the tuned search values.
The ZIP also contains pipeline/joblib models, ONNX files, CSV predictions, charts and the SQLite studies.
Raw input downloads are excluded from the ZIP to avoid duplicating a large source archive.
Use the frozen validation winner, not whichever finalist happens to score best on test.


In [ ]:
summary = dict(config=CONFIG, input_sha256=INPUT_SHA, dataset_sha256=DATASET_SHA,
    thresholds=THRESHOLDS, splits=split_manifest, baseline_validation=BASELINE_RESULTS,
    lazypredict=LAZY_RESULTS, top2_untuned=TOP2_RESULTS, top2_tuned=TUNED_RESULTS,
    winners_from_validation=WINNERS, final_fit=FINAL_FIT_REPORT,
    final_test=globals().get("TEST_RESULTS", {}), onnx_exports=globals().get("ONNX_REPORT", {}),
    final_test_evaluation_started=test_marker.exists() if "test_marker" in globals() else False,
    frozen_selection_sha256=FREEZE_SHA, probabilities="uncalibrated",
    limitations=["50-company present-day cohort: selection/survivorship bias",
        "Overlapping 20-session labels: metrics do not prove independent observations/significance",
        "Repeated Optuna selection may overfit the fixed validation holdout",
        "No approved Shariah screening, portfolio backtest or live investment validation",
        "Provider revisions and assumed historical after-close bar availability",
        "Regression risk predictions are raw and may be negative; check reported fraction"],
    notebook_execution="results generated by user runtime only")

# Sanitize using our robust local cleaner from Cell 4 before saving
save_json(OUT / "all_results.json", clean_json(summary))

inventory = [{"file": p.relative_to(OUT).as_posix(), "bytes": p.stat().st_size,
              "sha256": file_sha256(p)} for p in OUT.rglob("*")
             if p.is_file() and p.name not in {source_file.name, "artifact_manifest.json"}]
save_json(OUT / "artifact_manifest.json", clean_json(inventory))

zip_path = OUT.parent / f"{RUN_NAME}_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob("*"):
        if path.is_file() and path.resolve() != source_file.resolve():
            archive.write(path, arcname=path.relative_to(OUT))
print("Results:", OUT / "all_results.json")
print("Download:", zip_path, "| MB:", round(zip_path.stat().st_size / 1e6, 2))
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(zip_path))

## Reading the outputs

* `all_results.json`: complete results plus all effective model parameters and limitations.
* `reports/*_lazypredict.json`: full benchmark ranking, top five and failed/unscored families.
* `reports/*__optuna.json`: every trial's sampled parameters, effective parameters, scores, errors and duration.
* `frozen_selection.json`: winner choices made before final test access.
* `reports/*__final_test.json`: final test scores and year/ticker diagnostics.
* `models/*__final.joblib`: fitted scaler + estimator for both finalists and the baselines.
* `onnx/*.onnx` and matching JSON: converted pipelines with feature/class/output contracts.
* `reports/onnx_export_report.json`: conversion/parity status for every attempted model.
* `artifact_manifest.json`: hashes to verify the generated files.

Do not compare RMSE values across return and risk targets as if their scales were the same.
Macro-F1 is not accuracy. Classifier probabilities are not a percentage expected return.
ONNX inputs are engineered features, not raw OHLCV prices.

If data loading fails, check that you supplied the **separate readiness bundle**, not just a code ZIP.
If benchmark models fail, inspect the saved failure list and visible exception output.
If ONNX parity fails, retain the Python model and inspect the report before deployment; no export is certified by file existence.
All notebook outputs begin empty; no claimed performance numbers are fabricated.

### References
* [Project AGENTS.md](https://github.com/frager098/HalalifyAI/blob/integrate-data-processing/AGENTS.md)
* [Data readiness and data-transfer requirements](https://github.com/frager098/HalalifyAI/blob/integrate-data-processing/docs/data_readiness.md)
* [Repository feature implementation](https://github.com/frager098/HalalifyAI/blob/integrate-data-processing/src/features/core.py)
* [Repository targets and chronological splits](https://github.com/frager098/HalalifyAI/blob/integrate-data-processing/src/labels/targets.py)
* [LazyPredict source used by this notebook](https://github.com/shankarpandala/lazypredict/blob/v0.2.16/lazypredict/Supervised.py)
* [Optuna study API](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html)
* [sklearn-onnx conversion example](https://onnx.ai/sklearn-onnx/auto_examples/plot_convert_model.html)
